In [0]:
%pip install tqdm

In [0]:
import os
import pandas as pd
from tqdm import tqdm

In [0]:
# ── Unity Catalog location (must match Job 1's settings) ────────────────────
# See the README's "Key concepts" section for what catalog/schema mean.
CATALOG = "use1_prod_artemis_catalog_3718194974443840" #change
SCHEMA = "tier1_raw" #change
 
# Source table: the flight inventory table generated by Job 1 (1_Flight_table).
flight_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"
# Destination table: where this script will save the location-clip inventory.
TABLE_NAME_CLIP = f"{CATALOG}.{SCHEMA}.drone_location_clipped_table"

In [0]:
# Load the full flight inventory into a local pandas DataFrame so we can
# loop through it row by row.
flight_df = spark.read.table(flight_table).toPandas()
len(flight_df)

In [0]:
row_list = []
 
# ── Go through every flight and check its location-clip status ─────────────
for idx, row in tqdm(flight_df.iterrows(), total=len(flight_df)):
 
    # 1. Get the flight's base folder (same folder as its flight_details.json).
    flight_path = os.path.dirname(row['flight_metadata_path'])
 
    # DATABRICKS FIX: make sure 'os' can read the path correctly by converting
    # the "dbfs:/" prefix to "/dbfs/", which is what standard Python file
    # operations expect.
    if flight_path.startswith("dbfs:/"):
        flight_path = flight_path.replace("dbfs:/", "/dbfs/")
 
    # 2. Point to the folder where the clipped plot images would live.
    # (Currently this is just the flight's own folder — update this if your
    # clipped images are saved in a different subfolder.)
    plot_clipped_path = f"{flight_path}"
    plot_count = 0
 
    # 3. Count the clipped plot images IF the folder already exists.
    # We don't fail or skip the flight if it doesn't exist yet — we just
    # record a count of 0, so every flight still gets an inventory row.
    if os.path.exists(plot_clipped_path):
        files = os.listdir(plot_clipped_path)
        plot_count = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
 
    # 4. Always build the row dictionary, even if no clipped images exist yet.
    # This keeps a complete inventory of every flight and its current status,
    # which is what lets the orchestrator later figure out which flights are
    # still pending.
    row_dict = {
        'site': row['site'],
        'trial': row['trial'],
        'season': row['season'],
        'field':  row['field'],
        'location': row['location'],
        'mission': row['mission'],
        'flight_date': row['flight_date'],
 
        # Paths and metrics specific to this stage:
        'flight_metadata_path': row['flight_metadata_path'],
        'plot_clipped_path': plot_clipped_path,
        'plots_exist': plot_count > 0,   # Flag equivalent to 'ortho_exists' in the orthomosaic pipeline
        'plot_image_count': plot_count
    }
 
    row_list.append(row_dict)
 
# 5. Build the final DataFrame from all the collected rows.
location_df = pd.DataFrame(row_list)
 
print(f"Total flights processed for location clip inventory: {len(location_df)}")
 
# 6. Save the inventory permanently as a Delta table.
if len(location_df) > 0:
    display(location_df)
else:
    # If this is empty, it usually means Job 1 (flight table generation)
    # hasn't been run yet, since this script depends on that table existing.
    print("It's still empty. Check if you ran the flight_df table before running this.")

In [0]:
# Convert to a Spark DataFrame so it can be saved as a table in the catalog.
spark_df = spark.createDataFrame(location_df)
 
spark_df.printSchema()

In [0]:
# Write (overwrite) the location-clip inventory table. "mergeSchema" allows
# the table's schema to evolve if new columns are added in future runs.
spark_df.write.option("mergeSchema", "true").saveAsTable(TABLE_NAME_CLIP, mode="overwrite")